<a href="https://colab.research.google.com/github/kuds/rl-mujoco-tennis/blob/main/notebooks/%5BBall%20Bounce%5D%20Soft%20Actor-Critic%20%28SAC%29.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ball Bounce: Soft Actor-Critic (SAC)

Train a SAC agent to juggle a ball on a 6-DOF paddle. The custom `csv_header` / `info_row_fn` capture per-step touch and proximity sensors so rollout CSVs are inspectable.

## 1. Install

In [ ]:
!pip install -q "courtside-dynamics[train,notebooks] @ git+https://github.com/kuds/rl-mujoco-tennis"

## 2. Mount Google Drive (optional)

Set `USE_DRIVE = True` to persist checkpoints, TensorBoard logs, and replay videos under `MyDrive/courtside-dynamics/<run name>` so they survive a Colab runtime restart. With `USE_DRIVE = False` the run lives only in the ephemeral Colab VM.

In [ ]:
from courtside_dynamics.notebook_utils import mount_drive, resolve_log_dir

USE_DRIVE = False
if USE_DRIVE:
    mount_drive()

LOG_DIR = resolve_log_dir("BallBounce_SAC", use_drive=USE_DRIVE)
print("Logging to:", LOG_DIR)

## 3. Configure Colab GPU

Sets up EGL so MuJoCo can render off-screen on the Colab GPU. No-op outside of Colab.

In [ ]:
from courtside_dynamics.colab_setup import setup_colab
setup_colab()

## 4. Train

`train(cfg)` builds vectorized train + eval envs, attaches `EvalCallback`, `VideoRecordCallback`, and `InfoDictEvalCallback`, and runs SB3's `model.learn`. The best policy seen during evaluation is saved to `LOG_DIR/best_model.zip`.

In [ ]:
from courtside_dynamics.envs import BallBounceEnv
from courtside_dynamics.training import TrainConfig, train

env_fn = lambda: BallBounceEnv(render_mode="rgb_array", min_force=100.0)

cfg = TrainConfig(
    env_fn=env_fn,
    algo="SAC",
    total_timesteps=1_500_000,
    log_dir=LOG_DIR,
    name_prefix="ball_bounce_sac",
    eval_freq=25_000,
    csv_header=[
        "ball_velocity_x", "ball_velocity_y", "ball_velocity_z",
        "ball_accelerometer_x", "ball_accelerometer_y", "ball_accelerometer_z",
        "from_to_x1", "from_to_y1", "from_to_z1",
        "from_to_x2", "from_to_y2", "from_to_z2",
        "touch_sensor", "reward", "total_reward", "done",
    ],
    info_row_fn=lambda info, r, tr, d: [
        info["ball_velocity"], info["ball_accelerometer"],
        info["ball_to_paddle"], info["touch_sensor"], r, tr, d,
    ],
)
model = train(cfg)

## 5. Learning curves

Per-episode training rewards (left) come from `LOG_DIR/monitor/*.monitor.csv`. Deterministic eval rewards (right, mean +/- std) come from `LOG_DIR/evaluations.npz`.

In [ ]:
import os
from courtside_dynamics.notebook_utils import plot_learning_curve

plot_learning_curve(
    LOG_DIR,
    save_path=os.path.join(LOG_DIR, "learning_curve.png"),
)

## 6. Replay the best model

Loads `best_model.zip` from `LOG_DIR`, rolls it out deterministically, encodes the frames as MP4, and embeds the clip in this notebook.

In [ ]:
from courtside_dynamics.notebook_utils import (
    record_best_model_video,
    display_video,
)

video_path = record_best_model_video(
    LOG_DIR,
    env_fn,
    algo="SAC",
    video_length=750,
)
display_video(video_path)